# MovieLens 32M — Popularity and Genre CBF

Independent retrieval experiment. Popularity ranks Train-positive counts. Genre CBF uses normalized multi-hot movie genres and normalized Train-positive user profiles. Missing genres contribute no genre signal. Empty profiles receive a declared Popularity fallback.

All primary experiments use the entire external Train partition, Policy B (Train-observed movies), all-Train-rated seen filtering and the same frozen Validation users/labels. The default mode runs synthetic tests only. Saved outputs from the original `03_MovieLens32M_Retrieval_and_Ranking.ipynb` are preserved as a legacy reference and are not loaded by this notebook.


## 1. Environment and paths

Place this repository's readable `src/recsys/`, `configs/` and `tests/` directories in the Google Drive project directory below. No other notebook needs to have run in this runtime. The existing dataset and split artifacts must already exist for real experiments. The source path and file hashes are checked before imports; restart the runtime if a different or edited package is cached. Dependency requirements and installation instructions are in the project README; no automatic package installation runs here.


In [ ]:
from pathlib import Path
import sys
import importlib.util
RUN_MODE = "smoke_test"
if RUN_MODE == "smoke_test":
    import os
    for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[_thread_variable] = "1"  # bound synthetic CPU native-library concurrency before imports
SUPPORTED_MODES = ('smoke_test', 'full_train', 'load_evaluate', 'rank_snapshot')
if RUN_MODE not in SUPPORTED_MODES:
    raise ValueError(f"Unsupported mode {RUN_MODE!r}; choose {SUPPORTED_MODES}")
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "personal_projects/movielens_hybrid_recsys"
)
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")
SRC_DIR = PROJECT_DIR / "src"
if not (SRC_DIR / "recsys" / "__init__.py").is_file():
    raise FileNotFoundError(f"Readable source missing: {SRC_DIR / 'recsys'}. Copy or clone the repository there.")
import hashlib
_source_hash = hashlib.sha256()
for _file in sorted((SRC_DIR / "recsys").rglob("*.py")):
    _source_hash.update(str(_file.relative_to(SRC_DIR)).encode())
    _source_hash.update(_file.read_bytes())
_loaded = sys.modules.get("recsys")
if _loaded is not None:
    if (not Path(_loaded.__file__).resolve().is_relative_to(SRC_DIR.resolve()) or
        getattr(_loaded, "__notebook_source_hash__", None) != _source_hash.hexdigest()):
        raise RuntimeError("A stale/different recsys module is cached. Restart this runtime before importing.")
sys.path = [str(SRC_DIR)] + [p for p in sys.path if p != str(SRC_DIR) and "recsys_source" not in p]
import recsys
recsys.__notebook_source_hash__ = _source_hash.hexdigest()
from recsys.runtime import environment
from recsys.smoke import run_smoke
print("Source:", recsys.__file__)
print(environment("/content" if Path("/content").exists() else "/tmp"))


In [ ]:
if RUN_MODE == "smoke_test":
    print(run_smoke(PROJECT_DIR, 'baselines'))


## 2. Configuration

Edit the readable model JSON files and the explicit switches below. `full_train` uses every Train row. `load_evaluate` requires a checkpoint path and never fits a model. `rank_snapshot` requires an explicit UTC cutoff and produces later internal-Train candidates; it is not enabled by default and is not run as part of framework construction. Snapshot labels never select retriever epochs. Identical boundaries are required across all six snapshot channels before fusion.


In [ ]:
from recsys.config import load_model_config
from recsys.workflows import RetrievalExperiment
MODEL_NAMES = ('popularity', 'genre')
CONFIGS = {name: load_model_config(PROJECT_DIR / "configs", name) for name in MODEL_NAMES}
SAVED_CHECKPOINTS = {name: None for name in MODEL_NAMES}  # set explicit paths for load_evaluate
SNAPSHOT_CUTOFF_UTC = None  # e.g. an approved UTC boundary strictly inside external Train
SNAPSHOT_LABEL_END_UTC = None  # exclusive; None uses the external Validation start
DEVICE = 'cpu'
if RUN_MODE == "rank_snapshot":
    for _config in CONFIGS.values():
        _config["snapshot_cutoff_utc"] = SNAPSHOT_CUTOFF_UTC
        _config["snapshot_label_end_utc"] = SNAPSHOT_LABEL_END_UTC
experiments, candidate_outputs, checkpoint_paths = {}, {}, {}


## 3. Load and verify frozen splits

Verify the authoritative manifest, source/artifact checksums, schemas, Train/Validation counts and temporal boundaries. Expected 31,414,033 / 300,000 / 286,171 counts are integrity assertions, not model dimensions. Development does not deserialize Test labels. The cohort table is read only for Validation eligibility/reporting, never for model features.


In [ ]:
if RUN_MODE != "smoke_test":
    _shared_data = None
    for name in MODEL_NAMES:
        exp = RetrievalExperiment(PROJECT_DIR, name, CONFIGS[name], RUN_MODE,
                                  checkpoint=SAVED_CHECKPOINTS[name], device=DEVICE)
        print(exp.prepare(shared_data=_shared_data))
        _shared_data = exp.data  # only the explicitly grouped baselines share RAM within this notebook
        experiments[name] = exp


## 4. Prepare model-specific features

The shared context contains canonical IDs, all observed Train histories, sparse positive preferences, Train-only popularity and normalized movie/user genres. Only this notebook's requested model consumes its relevant features. No Validation-derived statistics, Test labels, tags or links enter the context.


In [ ]:
if RUN_MODE != "smoke_test":
    for name, exp in experiments.items():
        print(name, "observed/positive nnz:", exp.data.ctx.observed.nnz, exp.data.ctx.positive.nnz,
              "catalog:", int(exp.data.ctx.eligible.sum()), "cutoff:", exp.data.ctx.cutoff)


## 5. Initialize model

Inspect the single-model factory configuration, boundary and device. Parameter allocation occurs inside its fit implementation after resource guards; initialization does not launch training.


In [ ]:
if RUN_MODE != "smoke_test":
    for exp in experiments.values():
        print(exp.initialize())


## 6. Train or load exactly the requested model

A matching verified checkpoint is reused. Expensive work starts only in the selected real mode. CUDA is used for neural training; sparse CPU algorithms remain on CPU. Dynamic RAM/disk/VRAM guards stop infeasible stages rather than sample Train. Neural full-Train early stopping uses a deterministic Validation-only panel. Historical snapshots use fixed epochs and training loss only. No unrelated retriever is trained.


In [ ]:
if RUN_MODE != "smoke_test":
    for name, exp in experiments.items():
        print(name, exp.train())


## 7. Save checkpoint

Artifacts use `artifacts/<model>/<run_id>/checkpoint/`, with model files, mappings, configuration, training history, measured resources and a readable manifest. Writes are verified and immutable. Neural epoch state is saved separately; completed models resume automatically, while interrupted neural training restarts from its seed.


In [ ]:
if RUN_MODE != "smoke_test":
    for name, exp in experiments.items():
        checkpoint_paths[name] = exp.save_checkpoint()
        print(name, checkpoint_paths[name])


## 8. Validation retrieval

Every channel scores the full eligible unseen catalog in bounded chunks and emits at most Top-100 (fewer when filtering leaves fewer eligible movies). Raw scores and deterministic movie-ID tie breaking are preserved. `rank_snapshot` uses the later internal Train window instead of external Validation.


In [ ]:
if RUN_MODE != "smoke_test":
    for name, exp in experiments.items():
        candidate_outputs[name] = exp.evaluate()


## 9. Compute standard metrics

The shared evaluator reports user-macro Recall@20/50/100, HitRate@20/100, NDCG@20, cohort/bootstrap intervals, coverage and popularity concentration. No-hit queries remain in the fixed eligible-positive population. Candidate catalog exclusions and cold/profile fallback counts are explicit. Latency is batch-amortized, not online request latency.


In [ ]:
import json
import pandas as pd
from IPython.display import display
if RUN_MODE != "smoke_test":
    for name, path in candidate_outputs.items():
        summary = json.loads((path / "metrics.json").read_text())
        display(pd.DataFrame(summary["metrics"][name]))
        print(summary["eligibility"])


## 10. Save candidate outputs

Retrieval and metric computation already streamed the standardized file to `results/<validation|rank_train>/<model>/<run_id>/candidates/`. Required columns are `userId,movieId,score,rank,source`; `fallback` adds explicit provenance. The separate manifest records source hashes, split checksum, training cutoff, catalog/mappings, evaluation ID, K, seed, hyperparameters and candidate SHA256. Copy these explicit artifact paths into Notebook 08; no retraining is needed.


In [ ]:
if RUN_MODE != "smoke_test":
    from recsys.artifacts import read_artifact
    for name, path in candidate_outputs.items():
        manifest = read_artifact(path)
        print(name, "candidates:", path, "rows:", manifest["candidate_rows"], "run:", manifest["run_id"])


## 11. Resource and performance summary

Inspect training duration, sampled process RSS, CUDA allocator peaks, retrieval latency and file size. Checkpoint metadata retains package versions. No large weights or full candidate tables are printed.


In [ ]:
if RUN_MODE != "smoke_test":
    for name, exp in experiments.items():
        print(name, "training:", exp.training_profile,
              "candidate bytes:", (candidate_outputs[name] / "candidates.parquet").stat().st_size)
        history = json.loads((checkpoint_paths[name] / "training_history.json").read_text())
        if history["epochs"]:
            display(pd.DataFrame(history["epochs"]))


## 12. Limitations

Popularity ranks Train-positive counts. Genre CBF uses normalized multi-hot movie genres and normalized Train-positive user profiles. Missing genres contribute no genre signal. Empty profiles receive a declared Popularity fallback.

The benchmark recovers future observed high ratings, not exposure, watching, CTR or causal satisfaction. Missing-not-at-random feedback, submission-time bursts, weak unknown-pair supervision and temporal changes limit interpretation. Policy B excludes unseen-to-Train items and reports their lost future positives. Exact catalog scoring and sparse graph/index construction may still be slow. Source hashes and manifests must match for reuse; changed common protocol code requires a new run. No real MovieLens training or benchmark scores were produced while building this framework.
